# Leo — обучение своей модели с нуля (Google Colab)

Запускайте ячейки сверху вниз (кнопка ▶ слева). Перед началом: **Среда выполнения → Сменить тип среды → GPU** (T4).

Это маленькая модель, обученная с нуля: она научится писать простые связные тексты и коротко отвечать, но не сравнится с большими ИИ.


In [ ]:
# 1. Проверка видеокарты
!nvidia-smi -L

In [ ]:
# 2. Скачиваем код Leo и ставим зависимости
!git clone -q https://github.com/harringtonov-cloud/leo.ai
%cd leo.ai/leo-model
!pip install -q tokenizers datasets

In [ ]:
# 3. Самопроверка (должно закончиться словами "All checks passed.")
!python smoke_test.py

In [ ]:
# 4. Данные: короткие английские истории (TinyStories) + обучение токенизатора
!python prepare.py --hf roneneldan/TinyStories --max-docs 200000 --out data

In [ ]:
# 5. Предобучение (~15 млн параметров). Следите, чтобы val loss падал.
!python train.py --preset tiny --batch-size 32 --grad-accum 2 --max-steps 3000 --eval-interval 500

In [ ]:
# 6. Попробовать модель: она продолжает текст
import torch
from leo.model import Config, GPT
from leo.tokenizer import LeoTokenizer
from leo.generate import generate_stream

def load(path):
    ck = torch.load(path, map_location="cpu")
    m = GPT(Config(**ck["config"])); m.load_state_dict(ck["model"])
    return m.to("cuda" if torch.cuda.is_available() else "cpu").eval()

tok = LeoTokenizer("data/tokenizer.json")
model = load("out/ckpt.pt")
dev = next(model.parameters()).device
prompt = "Once upon a time"
ids = [t for t in generate_stream(model, tok.encode(prompt), 120, 0.8, 50, 0.95, {tok.ids["<|eos|>"]}, dev)]
print(prompt + tok.decode(ids))

In [ ]:
# 7. Обучение диалогу. Здесь всего 3 примера, только чтобы проверить, что всё работает.
# Для настоящего результата загрузите свой файл JSONL с тысячами диалогов (см. examples/sft_sample.jsonl).
!python sft.py --data examples/sft_sample.jsonl --epochs 30 --batch-size 3

In [ ]:
# 8. Спросить обученную модель в формате чата
model = load("sft_out/ckpt.pt")
dev = next(model.parameters()).device
msgs = [{"role": "user", "content": "Who are you?"}]
ids = [t for t in generate_stream(model, tok.encode_chat(msgs, True), 80, 0.5, 20, 0.9,
                                  {tok.ids["<|end|>"], tok.ids["<|eos|>"]}, dev)]
print(tok.decode(ids))

In [ ]:
# 9. Сохранить результат на Google Диск, иначе Colab всё удалит при закрытии
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/leo && cp -r out sft_out data/tokenizer.json /content/drive/MyDrive/leo/ 2>/dev/null
print("Сохранено в Google Диск → папка leo")

**Дальше:** Colab не подходит для постоянной работы сервера. Чтобы подключить Leo к сайту, модель нужно запустить на своём сервере (см. README в `leo-model`).